<a href="https://colab.research.google.com/github/lithinsaics24-blip/ADA-LAB/blob/main/ai_week_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Implementation of the Truth-Table Enumeration Algorithm

class Sentence:
    """Base class for propositional logic sentences."""
    def evaluate(self, model):
        raise NotImplementedError

    def symbols(self):
        raise NotImplementedError

class Symbol(Sentence):
    """A propositional symbol (e.g., P, Q, R)."""
    def __init__(self, name):
        self.name = name

    def evaluate(self, model):
        return model.get(self.name, False)

    def symbols(self):
        return {self.name}

    def __repr__(self):
        return self.name

class Not(Sentence):
    """Negation: ~P"""
    def __init__(self, op):
        self.op = op

    def evaluate(self, model):
        return not self.op.evaluate(model)

    def symbols(self):
        return self.op.symbols()

    def __repr__(self):
        return f"~({self.op})"

class And(Sentence):
    """Conjunction: P & Q"""
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def evaluate(self, model):
        return self.left.evaluate(model) and self.right.evaluate(model)

    def symbols(self):
        return self.left.symbols().union(self.right.symbols())

    def __repr__(self):
        return f"({self.left} & {self.right})"

class Or(Sentence):
    """Disjunction: P | Q"""
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def evaluate(self, model):
        return self.left.evaluate(model) or self.right.evaluate(model)

    def symbols(self):
        return self.left.symbols().union(self.right.symbols())

    def __repr__(self):
        return f"({self.left} | {self.right})"

class Implies(Sentence):
    """Implication: P => Q (equivalent to ~P | Q)"""
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def evaluate(self, model):
        return (not self.left.evaluate(model)) or self.right.evaluate(model)

    def symbols(self):
        return self.left.symbols().union(self.right.symbols())

    def __repr__(self):
        return f"({self.left} => {self.right})"

class Biconditional(Sentence):
    """Biconditional: P <=> Q"""
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def evaluate(self, model):
        return self.left.evaluate(model) == self.right.evaluate(model)

    def symbols(self):
        return self.left.symbols().union(self.right.symbols())

    def __repr__(self):
        return f"({self.left} <=> {self.right})"


In [ ]:
def tt_entails(KB, alpha):
    """Determine if the Knowledge Base (KB) entails query (alpha)."""
    # Get all unique symbols from KB and alpha
    kb_symbols = set()
    for sentence in KB:
        kb_symbols = kb_symbols.union(sentence.symbols())
    symbols = list(kb_symbols.union(alpha.symbols()))

    print(f"Symbols involved in KB & Alpha: {symbols}\n")
    print(f"{' | '.join(symbols)} | KB | Alpha | Status")
    print("-" * (len(symbols)*4 + 25))

    return tt_check_all(KB, alpha, symbols, {})

def tt_check_all(KB, alpha, symbols, model):
    """Recursively enumerate all truth assignments and check entailment."""
    if not symbols:
        # Evaluate KB as conjunction of all sentences in it
        kb_true = all(sentence.evaluate(model) for sentence in KB)
        alpha_true = alpha.evaluate(model)

        # Print the row of the truth table
        vals = [str(model[sym])[0] for sym in model]
        status = "Valid" if (not kb_true or alpha_true) else "Fails (Counter-example!)"
        print(f" {' | '.join(vals)} |  {str(kb_true)[0]} |   {str(alpha_true)[0]}   | {status}")

        if kb_true:
            return alpha_true
        else:
            return True
    else:
        P = symbols[0]
        rest = symbols[1:]

        # Check both possibilities (True/False) for current symbol
        # We need both paths to return True for entailment to hold
        extend_true = tt_check_all(KB, alpha, rest, {**model, P: True})
        extend_false = tt_check_all(KB, alpha, rest, {**model, P: False})
        return extend_true and extend_false

In [ ]:
# Let's test with a classic example:
# KB:
# 1. P => Q
# 2. P
# Query (Alpha): Q
# This is Modus Ponens, Q should be entailed (True).

P = Symbol('P')
Q = Symbol('Q')

KB1 = [
    Implies(P, Q),
    P
]
Alpha1 = Q

print("Evaluating Test Case 1: Modus Ponens (P => Q, P) |= Q")
result1 = tt_entails(KB1, Alpha1)
print(f"\nDoes KB entail Alpha? {result1}\n")

Evaluating Test Case 1: Modus Ponens (P => Q, P) |= Q
Symbols involved in KB & Alpha: ['Q', 'P']

Q | P | KB | Alpha | Status
---------------------------------
 T | T |  T |   T   | Valid
 T | F |  F |   T   | Valid
 F | T |  F |   F   | Valid
 F | F |  F |   F   | Valid

Does KB entail Alpha? True



In [ ]:
# Let's test a case where entailment fails:
# KB:
# 1. P | Q
# 2. ~P
# Query (Alpha): R

R = Symbol('R')
KB2 = [
    Or(P, Q),
    Not(P)
]
Alpha2 = R

print("Evaluating Test Case 2: (P | Q, ~P) |= R")
result2 = tt_entails(KB2, Alpha2)
print(f"\nDoes KB entail Alpha? {result2}\n")

Evaluating Test Case 2: (P | Q, ~P) |= R
Symbols involved in KB & Alpha: ['Q', 'P', 'R']

Q | P | R | KB | Alpha | Status
-------------------------------------
 T | T | T |  F |   T   | Valid
 T | T | F |  F |   F   | Valid
 T | F | T |  T |   T   | Valid
 T | F | F |  T |   F   | Fails (Counter-example!)
 F | T | T |  F |   T   | Valid
 F | T | F |  F |   F   | Valid
 F | F | T |  F |   T   | Valid
 F | F | F |  F |   F   | Valid

Does KB entail Alpha? False

